In [ ]:
import gradio as gr
import torch
import os
import requests
import logging
from PIL import Image, ImageEnhance
import numpy as np
import gc
from datetime import datetime
import re
import shutil
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_videoclips, CompositeAudioClip
from pydub import AudioSegment
from gtts import gTTS
import random
import time
import imageio

# Section 1: Setup and Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("/content/error_log.txt"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Global variable for preloaded model
base_pipeline = None
model_loaded = False

# Directories (Colab-specific)
base_dir = "/content"
for dir_path in ['outputs', 'temp', 'temp/frames', 'temp/gifs', 'temp/audio']:
    os.makedirs(os.path.join(base_dir, dir_path), exist_ok=True)

def clear_cuda_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        logger.info(f"CUDA memory cleared. Usage: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

def generate_gif_frames(prompt, negative_prompt, num_frames=12, num_inference_steps=25, guidance_scale=7.5, seed=None):
    """Generate a sequence of frames forming a short animated GIF."""
    try:
        global base_pipeline
        height, width = 768, 768  # High resolution for cinematic quality
        seed = seed if seed != -1 else random.randint(0, 1000000)
        generator = torch.Generator(device="cuda").manual_seed(seed)
        frames = []

        for frame_idx in range(num_frames):
            # Add slight variation to prompt for animation effect (e.g., zoom/movement)
            variation_prompt = f"{prompt}, frame {frame_idx+1} of {num_frames}, subtle motion, photorealistic, ultra-detailed, cinematic lighting"
            with torch.no_grad():
                image = base_pipeline(
                    prompt=variation_prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale,
                    width=width,
                    height=height,
                    generator=generator
                ).images[0]
            image = ImageEnhance.Sharpness(image).enhance(1.5)
            image = ImageEnhance.Contrast(image).enhance(1.2)
            frames.append(image)
            clear_cuda_memory()

        return frames, seed
    except Exception as e:
        logger.error(f"GIF frame generation error: {e}")
        blank = Image.new('RGB', (768, 768), 'gray')
        return [blank] * num_frames, seed

# Section 2: Preload Advanced Model (SDXL)
def preload_model():
    global base_pipeline, model_loaded
    try:
        if not torch.cuda.is_available():
            raise Exception("L4 GPU not detected. Enable GPU runtime.")

        logger.info("Preloading SDXL base model...")
        base_pipeline = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            use_safetensors=True,
            variant="fp16"
        )
        base_pipeline.scheduler = DPMSolverMultistepScheduler.from_config(base_pipeline.scheduler.config)
        base_pipeline.to("cuda")
        base_pipeline.enable_attention_slicing()

        # Test generation to ensure model works
        logger.info("Testing SDXL model with a sample GIF frame...")
        with torch.no_grad():
            test_image = base_pipeline(
                "Test cinematic scene, photorealistic, ultra-detailed",
                num_inference_steps=25,
                guidance_scale=7.5,
                width=768,
                height=768
            ).images[0]
        test_path = "/content/temp/test_image.png"
        test_image.save(test_path)
        if os.path.exists(test_path):
            logger.info(f"Test image preloaded at {test_path}")
            model_loaded = True
            return "SDXL preloaded successfully!"
        raise Exception("Test image generation failed")
    except Exception as e:
        logger.error(f"Preload error: {e}")
        return f"Failed to preload model: {str(e)}"

# Section 3: Storyboard Parsing (Real-Time from User Upload)
def parse_storyboard(storyboard_file):
    try:
        with open(storyboard_file, 'r', encoding='utf-8') as file:
            storyboard_text = file.read()
        scenes = []
        narration = ""
        sections = storyboard_text.split("---") if "---" in storyboard_text else storyboard_text.split("\n\n")
        for i, section in enumerate(sections[:5]):
            lines = section.strip().split('\n')
            desc = lines[0].strip() if lines else f"Scene {i+1}"
            prompt = next((l.split(":", 1)[1].strip() for l in lines if l.startswith("Image:") or l.startswith("Prompt:")), f"{desc}, cinematic, ultra-detailed, 4K")
            scenes.append({
                "time": f"{i*12}-{(i+1)*12}s",
                "description": desc,
                "prompt": prompt
            })
            narration += prompt + " "  # Use prompt for narration continuity
        while len(scenes) < 5:
            idx = len(scenes)
            default_prompt = f"Scene {idx+1}, cinematic, ultra-detailed, 4K"
            scenes.append({
                "time": f"{idx*12}-{(idx+1)*12}s",
                "description": f"Scene {idx+1}",
                "prompt": default_prompt
            })
            narration += default_prompt + " "
        logger.info("Storyboard parsed successfully from uploaded file")
        return scenes, narration.strip()
    except Exception as e:
        logger.error(f"Storyboard parsing error: {e}")
        return None, None

# Section 4: Voiceover Generation (Using Image Prompt Text with gTTS)
def generate_voiceover(scenes, output_path="/content/temp/audio/voiceover.mp3"):
    try:
        combined = AudioSegment.empty()
        scene_duration = 12000  # 12 seconds per scene

        for i, scene in enumerate(scenes[:5]):
            # Use the image prompt text for voiceover
            narr = scene["prompt"]
            if not narr.strip():
                narr = f"Scene {i+1} image description placeholder."
            temp_path = f"/content/temp/audio/scene_{i}.mp3"
            tts = gTTS(text=narr, lang='en', slow=False)
            tts.save(temp_path)
            if os.path.exists(temp_path):
                audio = AudioSegment.from_file(temp_path).normalize().fade_in(300).fade_out(300)
                audio = audio.speedup(playback_speed=1.05)  # Slight speedup for natural flow
                if len(audio) > scene_duration:
                    audio = audio[:scene_duration]
                elif len(audio) < scene_duration:
                    audio += AudioSegment.silent(duration=scene_duration - len(audio))
                combined += audio
                os.remove(temp_path)
            else:
                logger.error(f"Failed to generate narration for scene {i+1}")
                combined += AudioSegment.silent(duration=scene_duration)

        combined = combined.normalize().apply_gain(+5)  # Boost volume for clarity
        combined.export(output_path, format="mp3", bitrate="192k")  # High bitrate for quality
        if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
            logger.info(f"Voiceover generated at {output_path} from image prompts")
            return output_path
        raise Exception("Voiceover generation failed")
    except Exception as e:
        logger.error(f"Voiceover error: {e}")
        AudioSegment.silent(duration=60000).export(output_path, format="mp3")
        return output_path

# Section 5: Background Music (Using Bensound)
def generate_background_music(mood="cinematic", output_path="/content/temp/audio/background.mp3"):
    try:
        silent = AudioSegment.silent(duration=60000)
        silent.export(output_path, format="mp3")
        music_urls = {
            "cinematic": "https://www.bensound.com/bensound-music/bensound-slowmotion.mp3",
            "dramatic": "https://www.bensound.com/bensound-music/bensound-tenderness.mp3",
            "upbeat": "https://www.bensound.com/bensound-music/bensound-ukulele.mp3",
            "suspense": "https://www.bensound.com/bensound-music/bensound-instinct.mp3",
            "emotional": "https://www.bensound.com/bensound-music/bensound-onceagain.mp3"
        }
        yt_url = music_urls.get(mood.lower(), music_urls["cinematic"])
        response = requests.get(yt_url)
        if response.status_code == 200:
            with open("/content/temp/audio/temp_music.mp3", 'wb') as f:
                f.write(response.content)
            music = AudioSegment.from_file("/content/temp/audio/temp_music.mp3")[:60000].normalize().apply_gain(-8)
            music.export(output_path, format="mp3", bitrate="192k")
            os.remove("/content/temp/audio/temp_music.mp3")
            logger.info(f"Music ({mood}) at {output_path}")
            return output_path
        logger.warning("Using silent track")
        return output_path
    except Exception as e:
        logger.error(f"Music error: {e}")
        return output_path

# Section 6: GIF and Video Assembly
def create_scene_gif(frames, scene_idx, duration=12, fps=12):
    """Create a GIF for a single scene."""
    try:
        gif_path = f"/content/temp/gifs/scene_{scene_idx}.gif"
        imageio.mimsave(gif_path, frames, duration=duration/len(frames), loop=0)
        if os.path.exists(gif_path):
            logger.info(f"GIF created for scene {scene_idx} at {gif_path}")
            return gif_path
        raise Exception(f"GIF creation failed for scene {scene_idx}")
    except Exception as e:
        logger.error(f"GIF creation error: {e}")
        return None

def concatenate_gifs_to_video(gif_paths, output_path="/content/temp/video.mp4", fps=12):
    """Concatenate GIFs into a single video."""
    try:
        clips = [VideoFileClip(gif_path) for gif_path in gif_paths if gif_path and os.path.exists(gif_path)]
        if len(clips) != 5:
            raise ValueError(f"Expected 5 GIF clips, got {len(clips)}")
        final_clip = concatenate_videoclips(clips, method="compose")
        final_clip.write_videofile(output_path, codec="libx264", audio_codec="aac", fps=fps, verbose=False, logger=None)
        final_clip.close()
        for clip in clips:
            clip.close()
        if os.path.exists(output_path):
            logger.info(f"Video created from GIFs at {output_path}")
            return output_path
        raise Exception("Video creation failed")
    except Exception as e:
        logger.error(f"Video creation from GIFs error: {e}")
        return None

def add_audio_to_video(video_path, voiceover_path, music_path, output_path="/content/outputs/final_video.mp4"):
    """Add voiceover and music to the video."""
    try:
        video_clip = VideoFileClip(video_path)
        audio_tracks = []

        # Add voiceover
        if voiceover_path and os.path.exists(voiceover_path) and os.path.getsize(voiceover_path) > 1000:
            voiceover = AudioFileClip(voiceover_path)
            if voiceover.duration > video_clip.duration:
                voiceover = voiceover.subclip(0, video_clip.duration)
            voiceover = voiceover.volumex(1.5)  # Increase volume for clarity
            audio_tracks.append(voiceover)
            logger.info(f"Voiceover added from {voiceover_path}")
        else:
            logger.warning(f"Voiceover file invalid or missing: {voiceover_path}")

        # Add background music
        if music_path and os.path.exists(music_path) and os.path.getsize(music_path) > 1000:
            music = AudioFileClip(music_path)
            if music.duration < video_clip.duration:
                repeats = int(video_clip.duration / music.duration) + 1
                music = AudioFileClip(music_path * repeats).subclip(0, video_clip.duration)
            music = music.volumex(0.4)  # Background volume
            audio_tracks.append(music)
            logger.info(f"Music added from {music_path}")
        else:
            logger.warning(f"Music file invalid or missing: {music_path}")

        if audio_tracks:
            final_audio = CompositeAudioClip(audio_tracks)
            video_with_audio = video_clip.set_audio(final_audio)
            video_with_audio.write_videofile(output_path, codec="libx264", audio_codec="aac", fps=video_clip.fps, verbose=False, logger=None)
            video_with_audio.close()
            logger.info(f"Final video with audio at {output_path}")
        else:
            shutil.copy(video_path, output_path)
            logger.warning("No valid audio tracks; video saved without audio")

        video_clip.close()
        return output_path
    except Exception as e:
        logger.error(f"Audio addition error: {e}")
        if os.path.exists(video_path):
            shutil.copy(video_path, output_path)
            return output_path
        return None

# Section 7: Main Video Generation Function
def generate_cinematic_video(storyboard_file, negative_prompt, num_inference_steps, guidance_scale, seed, music_mood, progress=gr.Progress()):
    global model_loaded
    try:
        start_time = time.time()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_output_path = f"/content/outputs/video_{timestamp}.mp4"

        # Preload model if not already loaded
        if not model_loaded:
            progress(0.05, desc="Preloading SDXL model")
            preload_status = preload_model()
            if "Failed" in preload_status:
                raise Exception(f"Model preloading failed: {preload_status}")

        progress(0.1, desc="Parsing storyboard")
        scenes, _ = parse_storyboard(storyboard_file.name)  # Use .name to get filepath
        if not scenes:
            raise Exception("Failed to parse storyboard")
        storyboard_display = "# Storyboard\n\n" + "\n\n".join([f"## Scene {i+1}: {s['time']}\n{s['description']}\nImage: {s['prompt']}" for i, s in enumerate(scenes)])

        progress(0.2, desc="Generating voiceover from image prompts")
        voiceover_path = generate_voiceover(scenes)  # Use image prompts for voiceover

        progress(0.3, desc="Adding background music")
        music_path = generate_background_music(mood=music_mood)

        progress(0.4, desc="Generating GIFs with preloaded SDXL")
        gif_paths = []
        seed_used = seed
        for idx, scene in enumerate(scenes):
            progress(0.4 + (0.4 * idx / len(scenes)), desc=f"Generating GIF for scene {idx+1}")
            frames, seed_info = generate_gif_frames(
                scene["prompt"],
                negative_prompt,
                num_frames=12,  # 12 frames per GIF, ~1 second at 12 FPS
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                seed=seed_used
            )
            seed_used = seed_info  # Use the same seed for consistency or update for variation
            gif_path = create_scene_gif(frames, idx)
            if gif_path:
                gif_paths.append(gif_path)

        if len(gif_paths) != 5:
            raise Exception(f"Generated {len(gif_paths)} GIFs, expected 5")

        progress(0.8, desc="Concatenating GIFs into video")
        video_path = concatenate_gifs_to_video(gif_paths)
        if not video_path:
            raise Exception("GIF concatenation failed")

        progress(0.9, desc="Adding audio")
        final_video = add_audio_to_video(video_path, voiceover_path, music_path, final_output_path)
        if not final_video:
            raise Exception("Final video assembly failed")

        progress(0.95, desc="Cleaning up")
        for file in gif_paths + [video_path, voiceover_path, music_path]:
            if file and os.path.exists(file):
                os.remove(file)

        elapsed_time = time.time() - start_time
        logger.info(f"Video generated in {elapsed_time:.2f} seconds")
        progress(1.0, desc="Complete!")

        return final_video, storyboard_display, str(seed_used)
    except Exception as e:
        logger.error(f"Video generation error: {e}")
        return None, f"Error: {str(e)}", "Failed"

# Section 8: Gradio Interface
def create_interface():
    with gr.Blocks(title="Cinematic AI Video Generator with SDXL GIFs") as app:
        gr.Markdown("# Final Year Project: Cinematic AI Video Generator with SDXL GIFs\nCreate a 1-minute cinematic video with animated GIF scenes and voiceover from image prompts in ~5-6 minutes using Colab L4 GPU (24 GB)")
        with gr.Row():
            with gr.Column():
                gr.Markdown("## Input")
                storyboard_file = gr.File(label="Upload Storyboard Text File", file_types=[".txt"])
                gr.Markdown("**Format**: `Scene X: [desc]\nImage: [prompt]` (Voiceover uses Image prompt text)")
                negative_prompt = gr.Textbox(label="Negative Prompt", value="blurry, bad quality, deformed, ugly, low resolution, cartoonish, abstract", lines=2)
                num_inference_steps = gr.Slider(10, 50, 25, step=5, label="Inference Steps (Higher = More Detail)")
                guidance_scale = gr.Slider(1.0, 15.0, 7.5, step=0.5, label="Guidance Scale (Higher = More Prompt Adherence)")
                seed = gr.Number(label="Seed (-1 for random)", value=-1)
                music_mood = gr.Radio(["cinematic", "dramatic", "upbeat", "suspense", "emotional"], label="Music Mood", value="cinematic")
                create_btn = gr.Button("Generate 1-Minute Video", variant="primary")
            with gr.Column():
                gr.Markdown("## Output")
                video_output = gr.Video(label="Generated 1-Minute Cinematic Video")
                storyboard_output = gr.Markdown(label="Parsed Storyboard")
                seed_output = gr.Textbox(label="Used Seed")

        create_btn.click(
            fn=generate_cinematic_video,
            inputs=[storyboard_file, negative_prompt, num_inference_steps, guidance_scale, seed, music_mood],
            outputs=[video_output, storyboard_output, seed_output]
        )

        gr.Markdown("""
        ### Instructions:
        1. Upload a `.txt` file with your storyboard.
        2. The `Image:` text will be used for both animated GIF generation and voiceover.
        3. Click "Generate 1-Minute Video" (~5-6 minutes using preloaded SDXL).
        4. Download from `/content/outputs/`.
        - Each scene becomes a 12-frame GIF for a cinematic effect.
        - Models are preloaded on startup for efficiency.
        - Ensure GPU runtime is enabled (L4 recommended).
        """)

    return app

if __name__ == "__main__":
    # Preload model at startup
    preload_model()
    app = create_interface()
    app.launch(share=True)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
!pip install torch
!pip install diffusers
!pip install gradio
!pip install gradio torch diffusers moviepy pydub gtts
!pip install gradio torch diffusers transformers pyttsx3 moviepy pydub pillow numpy requests pytube --quiet
!apt-get update -qq
!apt-get install -y ffmpeg espeak -qq
!pip install gradio torch torchvision Pillow requests numpy opencv-python diffusers transformers moviepy pydub gTTS tqdm ffmpeg-python accelerate sentencepiece soundfile librosa
!pip install pytube

# Import statements from both codes
import numpy as np
import torch
import gradio as gr
from diffusers import I2VGenXLPipeline, StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_gif, load_image
import os
import time
import tempfile
from PIL import Image as PILImage, ImageEnhance
import requests
from io import BytesIO
from functools import lru_cache
import cv2
import moviepy.editor as mp
import json
import base64
from transformers import AutoProcessor, AutoModel
import matplotlib.pyplot as plt
import uuid
from collections import defaultdict
from pytube import YouTube
import logging
import gc
from datetime import datetime
import re
import shutil
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_videoclips, CompositeAudioClip
from pydub import AudioSegment
from gtts import gTTS
import random
import imageio

# Section 1: Setup and Configuration
# Logging configuration from second code
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("/content/error_log.txt"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Global variables
base_pipeline = None  # For SDXL from second code
sdxl_pipeline = None  # Renamed for clarity in combined code
model_loaded = False
user_analytics = defaultdict(int)  # From first code
generation_history = []  # From first code

# Directories (combined and standardized for Colab)
base_dir = "/content"
for dir_path in ['outputs', 'examples', 'user_gallery', 'temp_frames', 'exports', 'temp', 'temp/frames', 'temp/gifs', 'temp/audio']:
    os.makedirs(os.path.join(base_dir, dir_path), exist_ok=True)

# Load vision model for automatic prompt generation (from first code)
@lru_cache(maxsize=1)
def load_captioning_model():
    processor = AutoProcessor.from_pretrained("microsoft/git-base-coco")
    model = AutoModel.from_pretrained("microsoft/git-base-coco")
    return processor, model

# Function to generate caption for an image (from first code)
def generate_caption(image):
    try:
        processor, model = load_captioning_model()
        # Convert PIL Image to RGB if needed
        if image.mode != "RGB":
            image = image.convert("RGB")
        pixel_values = processor(images=image, return_tensors="pt").pixel_values
        with torch.no_grad():
            generated_ids = model.generate(pixel_values=pixel_values, max_length=50)
        generated_caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return generated_caption
    except Exception as e:
        print(f"Caption generation error: {str(e)}")
        logger.error(f"Caption generation error: {str(e)}")  # Added logging
        return "An interesting scene"

# Function to download image from URL (from first code)
def download_image(url):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        return PILImage.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        raise gr.Error(f"Failed to download image: {str(e)}")

# Function to load I2VGenXL pipeline (from first code)
@lru_cache(maxsize=1)
def load_pipeline():
    pipeline = I2VGenXLPipeline.from_pretrained(
        "ali-vilab/i2vgen-xl",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        variant="fp16" if torch.cuda.is_available() else None
    )
    if torch.cuda.is_available():
        pipeline.enable_model_cpu_offload()
    else:
        pipeline.to("cpu")
    return pipeline

# Preload SDXL model (from second code, renamed for clarity)
def preload_model():
    global sdxl_pipeline, model_loaded
    try:
        if not torch.cuda.is_available():
            raise Exception("L4 GPU not detected. Enable GPU runtime.")
        logger.info("Preloading SDXL base model...")
        sdxl_pipeline = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            use_safetensors=True,
            variant="fp16"
        )
        sdxl_pipeline.scheduler = DPMSolverMultistepScheduler.from_config(sdxl_pipeline.scheduler.config)
        sdxl_pipeline.to("cuda")
        sdxl_pipeline.enable_attention_slicing()
        # Test generation to ensure model works
        logger.info("Testing SDXL model with a sample GIF frame...")
        with torch.no_grad():
            test_image = sdxl_pipeline(
                "Test cinematic scene, photorealistic, ultra-detailed",
                num_inference_steps=25,
                guidance_scale=7.5,
                width=768,
                height=768
            ).images[0]
        test_path = "/content/temp/test_image.png"
        test_image.save(test_path)
        if os.path.exists(test_path):
            logger.info(f"Test image preloaded at {test_path}")
            model_loaded = True
            return "SDXL preloaded successfully!"
        raise Exception("Test image generation failed")
    except Exception as e:
        logger.error(f"Preload error: {e}")
        model_loaded = False
        return f"Failed to preload model: {str(e)}"

# Clear CUDA memory (from second code)
def clear_cuda_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        logger.info(f"CUDA memory cleared. Usage: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

# Video post-processing utilities (from first code)
def apply_video_effects(frames, effect_type="none", intensity=0.5):
    processed_frames = []
    for frame in frames:
        # Convert PIL to numpy for OpenCV
        img = np.array(frame)
        img = img[:, :, ::-1].copy()  # RGB to BGR
        if effect_type == "vintage":
            # Apply sepia tone effect
            sepia_kernel = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            sepia_img = cv2.transform(img, sepia_kernel)
            img = cv2.addWeighted(img, 1-intensity, sepia_img, intensity, 0)
            # Add random noise for film grain effect
            noise = np.random.randint(0, 50, img.shape, dtype=np.uint8)
            img = cv2.addWeighted(img, 0.9, noise, 0.1, 0)
        elif effect_type == "dream":
            # Apply dream-like glow effect
            blurred = cv2.GaussianBlur(img, (0, 0), 10)
            img = cv2.addWeighted(img, 1-intensity*0.5, blurred, intensity*0.5, 0)
            # Enhance colors
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            hsv[:,:,1] = hsv[:,:,1] * (1 + intensity*0.3)  # Increase saturation
            img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
        elif effect_type == "cinematic":
            # Apply letterbox
            h, w = img.shape[:2]
            bar_height = int(h * 0.1 * intensity)
            img[0:bar_height, :] = [0, 0, 0]
            img[h-bar_height:h, :] = [0, 0, 0]
            # Color grading
            img = cv2.convertScaleAbs(img, alpha=1+intensity*0.3, beta=-30*intensity)
        # Convert back to PIL for GIF export
        processed_frames.append(PILImage.fromarray(img[:, :, ::-1]))  # BGR to RGB
    return processed_frames

# Function to export as MP4 (from first code)
def frames_to_mp4(frames, output_path, fps=15):
    # Create a temporary directory for frames
    temp_dir = os.path.join("/content/temp_frames", str(uuid.uuid4()))
    os.makedirs(temp_dir, exist_ok=True)
    try:
        # Save frames as images
        frame_paths = []
        for i, frame in enumerate(frames):
            frame_path = os.path.join(temp_dir, f"frame_{i:04d}.png")
            frame.save(frame_path)
            frame_paths.append(frame_path)
        # Create video using MoviePy
        clip = mp.ImageSequenceClip(frame_paths, fps=fps)
        clip.write_videofile(output_path, codec="libx264", fps=fps)
        return output_path
    finally:
        # Clean up temporary files
        for file in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, file))
        os.rmdir(temp_dir)

# Main generation function for I2VGenXL (from first code)
def generate_video(
    input_image,
    prompt,
    negative_prompt,
    num_inference_steps,
    guidance_scale,
    effect_type,
    effect_intensity,
    export_format,
    seed,
    progress=gr.Progress()
):
    try:
        # Track usage
        user_analytics["total_generations"] += 1
        user_analytics[f"effect_{effect_type}"] += 1
        # Update progress
        progress(0, desc="Loading model...")
        # Load pipeline
        pipeline = load_pipeline()
        # Set random seed
        generator = None if seed == -1 else torch.manual_seed(seed)
        # Update progress
        progress(0.2, desc="Processing image and generating video...")
        # Generate frames
        result = pipeline(
            prompt=prompt,
            image=input_image,
            num_inference_steps=num_inference_steps,
            negative_prompt=negative_prompt,
            guidance_scale=guidance_scale,
            generator=generator,
        ).frames[0]
        # Apply post-processing effects
        progress(0.7, desc=f"Applying {effect_type} effect...")
        if effect_type != "none":
            processed_frames = apply_video_effects(result, effect_type, effect_intensity)
        else:
            processed_frames = result
        # Export the result
        progress(0.9, desc=f"Exporting to {export_format}...")
        timestamp = int(time.time())
        session_id = str(uuid.uuid4())[:8]
        if export_format == "gif":
            output_path = f"/content/outputs/i2v_output_{session_id}_{timestamp}.gif"
            export_to_gif(processed_frames, output_path)
        else:  # mp4
            output_path = f"/content/outputs/i2v_output_{session_id}_{timestamp}.mp4"
            frames_to_mp4(processed_frames, output_path)
        # Save to user gallery
        gallery_path = f"/content/user_gallery/i2v_output_{session_id}_{timestamp}.gif"
        export_to_gif(processed_frames, gallery_path)
        # Log generation
        generation_history.append({
            "timestamp": timestamp,
            "prompt": prompt,
            "effect": effect_type,
            "path": gallery_path
        })
        clear_cuda_memory()  # Added memory cleanup
        # Return paths for display
        return output_path, output_path, f"✅ Generation complete! Prompt: '{prompt}'"
    except Exception as e:
        user_analytics["errors"] += 1
        logger.error(f"I2VGenXL generation error: {str(e)}")
        return None, None, f"❌ Error: {str(e)}"

# Function to process URL inputs (from first code)
def process_url_input(url, prompt, negative_prompt, num_inference_steps, guidance_scale,
                     effect_type, effect_intensity, export_format, seed, progress=gr.Progress()):
    try:
        progress(0, desc="Downloading image...")
        image = download_image(url)
        return generate_video(image, prompt, negative_prompt, num_inference_steps, guidance_scale,
                             effect_type, effect_intensity, export_format, seed, progress)
    except Exception as e:
        return None, None, f"❌ Error with URL: {str(e)}"

# Function for batch processing (from first code)
def batch_process(
    input_images,
    prompts,
    negative_prompt,
    num_inference_steps,
    guidance_scale,
    effect_type,
    effect_intensity,
    export_format,
    seed,
    progress=gr.Progress()
):
    results = []
    messages = []
    if len(input_images) != len(prompts.split("\n")):
        return [], "❌ Error: Number of images must match number of prompts (one prompt per line)"
    prompt_list = [p.strip() for p in prompts.split("\n") if p.strip()]
    for i, (img, prompt) in enumerate(zip(input_images, prompt_list)):
        progress((i / len(input_images)), desc=f"Processing image {i+1}/{len(input_images)}")
        try:
            progress(0.1 + (i / len(input_images) * 0.9), desc=f"Generating video {i+1}/{len(input_images)}")
            output_path, _, msg = generate_video(
                img, prompt, negative_prompt,
                num_inference_steps, guidance_scale,
                effect_type, effect_intensity, export_format,
                seed if seed != -1 else int(time.time() + i)
            )
            results.append(output_path)
            messages.append(f"Image {i+1}: {msg}")
        except Exception as e:
            messages.append(f"Image {i+1}: ❌ Error: {str(e)}")
    return results, "\n".join(messages)

# Function to generate suggested prompts based on the image (from first code)
def suggest_prompts(image):
    if image is None:
        return ""
    try:
        # Generate a base caption
        base_caption = generate_caption(image)
        # Create motion-specific variations
        variations = [
            f"{base_caption} with gentle movement",
            f"{base_caption} swaying slightly",
            f"Wind blowing through {base_caption}",
            f"{base_caption} with subtle animation",
            f"{base_caption} coming to life"
        ]
        return "\n".join(variations)
    except Exception as e:
        return f"Error generating suggestions: {str(e)}"

# Function to analyze generation history and create a chart (from first code)
def generate_analytics_chart():
    # Create a simple bar chart of effects used
    effect_counts = {}
    for key, value in user_analytics.items():
        if key.startswith("effect_"):
            effect_name = key.replace("effect_", "")
            if effect_name != "none":  # Skip the "none" effect for clarity
                effect_counts[effect_name] = value
    if not effect_counts:
        return None
    # Create the chart
    plt.figure(figsize=(10, 6))
    plt.bar(effect_counts.keys(), effect_counts.values(), color='skyblue')
    plt.title('Effects Usage')
    plt.xlabel('Effect Type')
    plt.ylabel('Count')
    plt.tight_layout()
    # Save to a temporary file
    chart_path = "/content/outputs/analytics_chart.png"
    plt.savefig(chart_path)
    plt.close()
    return chart_path

# Generate GIF frames for SDXL (from second code)
def generate_gif_frames(prompt, negative_prompt, num_frames=12, num_inference_steps=25, guidance_scale=7.5, seed=None):
    """Generate a sequence of frames forming a short animated GIF."""
    try:
        global sdxl_pipeline
        height, width = 768, 768  # High resolution for cinematic quality
        seed = seed if seed != -1 else random.randint(0, 1000000)
        generator = torch.Generator(device="cuda").manual_seed(seed)
        frames = []
        for frame_idx in range(num_frames):
            # Add slight variation to prompt for animation effect
            variation_prompt = f"{prompt}, frame {frame_idx+1} of {num_frames}, subtle motion, photorealistic, ultra-detailed, cinematic lighting"
            with torch.no_grad():
                image = sdxl_pipeline(
                    prompt=variation_prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale,
                    width=width,
                    height=height,
                    generator=generator
                ).images[0]
            image = ImageEnhance.Sharpness(image).enhance(1.5)
            image = ImageEnhance.Contrast(image).enhance(1.2)
            frames.append(image)
            clear_cuda_memory()
        return frames, seed
    except Exception as e:
        logger.error(f"GIF frame generation error: {e}")
        blank = Image.new('RGB', (768, 768), 'gray')
        return [blank] * num_frames, seed

# Storyboard parsing (from second code)
def parse_storyboard(storyboard_file):
    try:
        with open(storyboard_file, 'r', encoding='utf-8') as file:
            storyboard_text = file.read()
        scenes = []
        narration = ""
        sections = storyboard_text.split("---") if "---" in storyboard_text else storyboard_text.split("\n\n")
        for i, section in enumerate(sections[:5]):
            lines = section.strip().split('\n')
            desc = lines[0].strip() if lines else f"Scene {i+1}"
            prompt = next((l.split(":", 1)[1].strip() for l in lines if l.startswith("Image:") or l.startswith("Prompt:")), f"{desc}, cinematic, ultra-detailed, 4K")
            scenes.append({
                "time": f"{i*12}-{(i+1)*12}s",
                "description": desc,
                "prompt": prompt
            })
            narration += prompt + " "  # Use prompt for narration continuity
        while len(scenes) < 5:
            idx = len(scenes)
            default_prompt = f"Scene {idx+1}, cinematic, ultra-detailed, 4K"
            scenes.append({
                "time": f"{idx*12}-{(i+1)*12}s",
                "description": f"Scene {idx+1}",
                "prompt": default_prompt
            })
            narration += default_prompt + " "
        logger.info("Storyboard parsed successfully from uploaded file")
        return scenes, narration.strip()
    except Exception as e:
        logger.error(f"Storyboard parsing error: {e}")
        return None, None

# Voiceover generation (from second code)
def generate_voiceover(scenes, output_path="/content/temp/audio/voiceover.mp3"):
    try:
        combined = AudioSegment.empty()
        scene_duration = 12000  # 12 seconds per scene
        for i, scene in enumerate(scenes[:5]):
            # Use the image prompt text for voiceover
            narr = scene["prompt"]
            if not narr.strip():
                narr = f"Scene {i+1} image description placeholder."
            temp_path = f"/content/temp/audio/scene_{i}.mp3"
            tts = gTTS(text=narr, lang='en', slow=False)
            tts.save(temp_path)
            if os.path.exists(temp_path):
                audio = AudioSegment.from_file(temp_path).normalize().fade_in(300).fade_out(300)
                audio = audio.speedup(playback_speed=1.05)  # Slight speedup for natural flow
                if len(audio) > scene_duration:
                    audio = audio[:scene_duration]
                elif len(audio) < scene_duration:
                    audio += AudioSegment.silent(duration=scene_duration - len(audio))
                combined += audio
                os.remove(temp_path)
            else:
                logger.error(f"Failed to generate narration for scene {i+1}")
                combined += AudioSegment.silent(duration=scene_duration)
        combined = combined.normalize().apply_gain(+5)  # Boost volume for clarity
        combined.export(output_path, format="mp3", bitrate="192k")  # High bitrate for quality
        if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
            logger.info(f"Voiceover generated at {output_path} from image prompts")
            return output_path
        raise Exception("Voiceover generation failed")
    except Exception as e:
        logger.error(f"Voiceover error: {e}")
        AudioSegment.silent(duration=60000).export(output_path, format="mp3")
        return output_path

# Background music generation (from second code)
def generate_background_music(mood="cinematic", output_path="/content/temp/audio/background.mp3"):
    try:
        silent = AudioSegment.silent(duration=60000)
        silent.export(output_path, format="mp3")
        music_urls = {
            "cinematic": "https://www.bensound.com/bensound-music/bensound-slowmotion.mp3",
            "dramatic": "https://www.bensound.com/bensound-music/bensound-tenderness.mp3",
            "upbeat": "https://www.bensound.com/bensound-music/bensound-ukulele.mp3",
            "suspense": "https://www.bensound.com/bensound-music/bensound-instinct.mp3",
            "emotional": "https://www.bensound.com/bensound-music/bensound-onceagain.mp3"
        }
        yt_url = music_urls.get(mood.lower(), music_urls["cinematic"])
        response = requests.get(yt_url)
        if response.status_code == 200:
            with open("/content/temp/audio/temp_music.mp3", 'wb') as f:
                f.write(response.content)
            music = AudioSegment.from_file("/content/temp/audio/temp_music.mp3")[:60000].normalize().apply_gain(-8)
            music.export(output_path, format="mp3", bitrate="192k")
            os.remove("/content/temp/audio/temp_music.mp3")
            logger.info(f"Music ({mood}) at {output_path}")
            return output_path
        logger.warning("Using silent track")
        return output_path
    except Exception as e:
        logger.error(f"Music error: {e}")
        return output_path

# Create scene GIF (from second code)
def create_scene_gif(frames, scene_idx, duration=12, fps=12):
    """Create a GIF for a single scene."""
    try:
        gif_path = f"/content/temp/gifs/scene_{scene_idx}.gif"
        imageio.mimsave(gif_path, frames, duration=duration/len(frames), loop=0)
        if os.path.exists(gif_path):
            logger.info(f"GIF created for scene {scene_idx} at {gif_path}")
            return gif_path
        raise Exception(f"GIF creation failed for scene {scene_idx}")
    except Exception as e:
        logger.error(f"GIF creation error: {e}")
        return None

# Concatenate GIFs to video (from second code)
def concatenate_gifs_to_video(gif_paths, output_path="/content/temp/video.mp4", fps=12):
    """Concatenate GIFs into a single video."""
    try:
        clips = [VideoFileClip(gif_path) for gif_path in gif_paths if gif_path and os.path.exists(gif_path)]
        if len(clips) != 5:
            raise ValueError(f"Expected 5 GIF clips, got {len(clips)}")
        final_clip = concatenate_videoclips(clips, method="compose")
        final_clip.write_videofile(output_path, codec="libx264", audio_codec="aac", fps=fps, verbose=False, logger=None)
        final_clip.close()
        for clip in clips:
            clip.close()
        if os.path.exists(output_path):
            logger.info(f"Video created from GIFs at {output_path}")
            return output_path
        raise Exception("Video creation failed")
    except Exception as e:
        logger.error(f"Video creation from GIFs error: {e}")
        return None

# Add audio to video (from second code)
def add_audio_to_video(video_path, voiceover_path, music_path, output_path="/content/outputs/final_video.mp4"):
    """Add voiceover and music to the video."""
    try:
        video_clip = VideoFileClip(video_path)
        audio_tracks = []
        # Add voiceover
        if voiceover_path and os.path.exists(voiceover_path) and os.path.getsize(voiceover_path) > 1000:
            voiceover = AudioFileClip(voiceover_path)
            if voiceover.duration > video_clip.duration:
                voiceover = voiceover.subclip(0, video_clip.duration)
            voiceover = voiceover.volumex(1.5)  # Increase volume for clarity
            audio_tracks.append(voiceover)
            logger.info(f"Voiceover added from {voiceover_path}")
        else:
            logger.warning(f"Voiceover file invalid or missing: {voiceover_path}")
        # Add background music
        if music_path and os.path.exists(music_path) and os.path.getsize(music_path) > 1000:
            music = AudioFileClip(music_path)
            if music.duration < video_clip.duration:
                repeats = int(video_clip.duration / music.duration) + 1
                music = AudioFileClip(music_path * repeats).subclip(0, video_clip.duration)
            music = music.volumex(0.4)  # Background volume
            audio_tracks.append(music)
            logger.info(f"Music added from {music_path}")
        else:
            logger.warning(f"Music file invalid or missing: {music_path}")
        if audio_tracks:
            final_audio = CompositeAudioClip(audio_tracks)
            video_with_audio = video_clip.set_audio(final_audio)
            video_with_audio.write_videofile(output_path, codec="libx264", audio_codec="aac", fps=video_clip.fps, verbose=False, logger=None)
            video_with_audio.close()
            logger.info(f"Final video with audio at {output_path}")
        else:
            shutil.copy(video_path, output_path)
            logger.warning("No valid audio tracks; video saved without audio")
        video_clip.close()
        return output_path
    except Exception as e:
        logger.error(f"Audio addition error: {e}")
        if os.path.exists(video_path):
            shutil.copy(video_path, output_path)
            return output_path
        return None

# Main cinematic video generation function (from second code)
def generate_cinematic_video(storyboard_file, negative_prompt, num_inference_steps, guidance_scale, seed, music_mood, progress=gr.Progress()):
    global model_loaded
    try:
        start_time = time.time()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_output_path = f"/content/outputs/video_{timestamp}.mp4"
        # Preload model if not already loaded
        if not model_loaded:
            progress(0.05, desc="Preloading SDXL model")
            preload_status = preload_model()
            if "Failed" in preload_status:
                raise Exception(f"Model preloading failed: {preload_status}")
        progress(0.1, desc="Parsing storyboard")
        scenes, _ = parse_storyboard(storyboard_file.name)  # Use .name to get filepath
        if not scenes:
            raise Exception("Failed to parse storyboard")
        storyboard_display = "# Storyboard\n\n" + "\n\n".join([f"## Scene {i+1}: {s['time']}\n{s['description']}\nImage: {s['prompt']}" for i, s in enumerate(scenes)])
        progress(0.2, desc="Generating voiceover from image prompts")
        voiceover_path = generate_voiceover(scenes)  # Use image prompts for voiceover
        progress(0.3, desc="Adding background music")
        music_path = generate_background_music(mood=music_mood)
        progress(0.4, desc="Generating GIFs with preloaded SDXL")
        gif_paths = []
        seed_used = seed
        for idx, scene in enumerate(scenes):
            progress(0.4 + (0.4 * idx / len(scenes)), desc=f"Generating GIF for scene {idx+1}")
            frames, seed_info = generate_gif_frames(
                scene["prompt"],
                negative_prompt,
                num_frames=12,  # 12 frames per GIF, ~1 second at 12 FPS
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                seed=seed_used
            )
            seed_used = seed_info  # Use the same seed for consistency or update for variation
            gif_path = create_scene_gif(frames, idx)
            if gif_path:
                gif_paths.append(gif_path)
        if len(gif_paths) != 5:
            raise Exception(f"Generated {len(gif_paths)} GIFs, expected 5")
        progress(0.8, desc="Concatenating GIFs into video")
        video_path = concatenate_gifs_to_video(gif_paths)
        if not video_path:
            raise Exception("GIF concatenation failed")
        progress(0.9, desc="Adding audio")
        final_video = add_audio_to_video(video_path, voiceover_path, music_path, final_output_path)
        if not final_video:
            raise Exception("Final video assembly failed")
        progress(0.95, desc="Cleaning up")
        for file in gif_paths + [video_path, voiceover_path, music_path]:
            if file and os.path.exists(file):
                os.remove(file)
        elapsed_time = time.time() - start_time
        logger.info(f"Video generated in {elapsed_time:.2f} seconds")
        progress(1.0, desc="Complete!")
        return final_video, storyboard_display, str(seed_used)
    except Exception as e:
        logger.error(f"Video generation error: {e}")
        return None, f"Error: {str(e)}", "Failed"

# Example prompts for inspiration (from first code)
example_prompts = [
    "Papers floating in the air on a table in the library",
    "Leaves blowing in the wind on a sunny autumn day",
    "Ocean waves crashing against a rocky shore at sunset",
    "Butterflies flying around colorful flowers in a garden",
    "Snow falling gently in a quiet forest",
    "Clouds moving across a blue sky",
    "Rain falling on a window with city lights in the background",
    "Smoke rising from a campfire in the woods"
]

# Default negative prompt (from first code)
default_negative_prompt = "Distorted, discontinuous, ugly, blurry, low resolution, motionless, static, disfigured, disconnected limbs, ugly faces, incomplete arms"

# Example images (from first code)
example_images = [
    "https://huggingface.co/datasets/diffusers/docs-images/resolve/main/i2vgen_xl_images/img_0009.png",
    "https://huggingface.co/datasets/diffusers/docs-images/resolve/main/i2vgen_xl_images/img_0011.png",
    "https://huggingface.co/datasets/diffusers/docs-images/resolve/main/i2vgen_xl_images/img_0012.png",
    "https://huggingface.co/datasets/diffusers/docs-images/resolve/main/i2vgen_xl_images/img_0013.png",
]

# HTML components for the custom UI (from first code)
custom_header = """
<div class="custom-header">
    <div class="header-content">
        <h1>🎬 MotionMaster Pro</h1>
        <p class="subtitle">Advanced AI-Powered Image-to-Video Generation</p>
        <div class="header-badges">
            <span class="feature-badge">Premium</span>
            <span class="feature-badge new-badge">2025 Edition</span>
        </div>
    </div>
    <div class="animated-logo">
        <div class="circle"></div>
        <div class="square"></div>
        <div class="triangle"></div>
    </div>
</div>
"""

footer_html = """
<footer class="custom-footer">
    <div class="footer-content">
        <div class="footer-section">
            <h3>MotionMaster Pro</h3>
            <p>Powered by I2VGenXL & Advanced AI</p>
            <p>© 2025 All Rights Reserved</p>
            <div class="footer-social">
                <a href="#" aria-label="Twitter"><i class="fa fa-twitter"></i></a>
                <a href="#" aria-label="GitHub"><i class="fa fa-github"></i></a>
                <a href="#" aria-label="Discord"><i class="fa fa-discord"></i></a>
            </div>
        </div>
        <div class="footer-section">
            <h3>Resources</h3>
            <ul>
                <li><a href="https://huggingface.co/ali-vilab/i2vgen-xl" target="_blank">Model Documentation</a></li>
                <li><a href="https://docs.gradio.app/" target="_blank">Gradio Framework</a></li>
                <li><a href="#" target="_blank">User Guide</a></li>
                <li><a href="#" target="_blank">API Documentation</a></li>
                <li><a href="#" target="_blank">Community Showcase</a></li>
            </ul>
        </div>
        <div class="footer-section">
            <h3>Connect</h3>
            <ul>
                <li><a href="#" target="_blank">Twitter</a></li>
                <li><a href="#" target="_blank">GitHub</a></li>
                <li><a href="#" target="_blank">Discord Community</a></li>
                <li><a href="#" target="_blank">YouTube Channel</a></li>
                <li><a href="#" target="_blank">Support Forum</a></li>
            </ul>
        </div>
        <div class="footer-section">
            <h3>Newsletter</h3>
            <p>Stay updated with the latest features and improvements</p>
            <div class="newsletter-form">
                <input type="email" placeholder="Your email address" class="newsletter-input">
                <button class="newsletter-button">Subscribe</button>
            </div>
        </div>
    </div>
    <div class="footer-bar">
        <p>Created with ❤️ for AI Enthusiasts</p>
        <div class="footer-links">
            <a href="#" target="_blank">Privacy Policy</a> | 
            <a href="#" target="_blank">Terms of Service</a> | 
            <a href="#" target="_blank">Contact Us</a>
        </div>
    </div>
</footer>
"""

creative_toolkit_html = """
<div class="creative-toolkit">
    <h3>🎨 Creative Toolkit</h3>
    <div class="toolkit-content">
        <div class="toolkit-card">
            <h4>Motion Types</h4>
            <ul>
                <li><strong>Gentle:</strong> "swaying slightly", "subtle movement", "breathing effect"</li>
                <li><strong>Flowing:</strong> "flowing", "streaming", "cascading", "rippling"</li>
                <li><strong>Dynamic:</strong> "energetic", "vibrant", "pulsing", "bouncing"</li>
                <li><strong>Natural:</strong> "wind blowing", "waves crashing", "leaves rustling", "clouds drifting"</li>
                <li><strong>Cinematic:</strong> "panning", "zooming", "tracking", "rotating"</li>
            </ul>
        </div>
        <div class="toolkit-card">
            <h4>Effective Prompts</h4>
            <ul>
                <li>Be specific about the type of motion</li>
                <li>Mention direction: "from left to right", "upward", "circular"</li>
                <li>Describe speed: "slowly", "rapidly", "gently", "rhythmically"</li>
                <li>Include atmosphere: "dreamy", "energetic", "peaceful", "mysterious"</li>
                <li>Add lighting: "golden hour", "soft backlight", "dramatic shadows"</li>
            </ul>
        </div>
        <div class="toolkit-card">
            <h4>Style Examples</h4>
            <ul>
                <li><strong>Dreamlike:</strong> "ethereal", "floating", "hazy transitions"</li>
                <li><strong>Urban:</strong> "city pulse", "street motion", "neon accents"</li>
                <li><strong>Nature:</strong> "organic flow", "seasonal change", "wildlife movement"</li>
                <li><strong>Abstract:</strong> "geometric patterns", "color morphing", "particle flow"</li>
            </ul>
        </div>
    </div>
    <div class="toolkit-examples">
        <h4>Example Combinations</h4>
        <div class="example-tags">
            <span class="example-tag">"Gentle waves rolling onto shore at golden hour"</span>
            <span class="example-tag">"Vibrant autumn leaves swirling in wind"</span>
            <span class="example-tag">"Slow zoom on blooming flower with morning dew"</span>
            <span class="example-tag">"Urban traffic flowing through city at night with neon reflections"</span>
        </div>
    </div>
</div>
"""

ar_preview_html = """
<div class="ar-preview-section">
    <h3>🔮 AR Preview Mode</h3>
    <div class="ar-content">
        <div class="ar-description">
            <p>Experience how your animated videos will look in augmented reality environments. Enable AR preview to visualize your creations in real-world contexts.</p>
            <div class="feature-badge new-badge">NEW</div>
        </div>
        <div class="ar-preview-options">
            <div class="ar-option-card">
                <h4>Living Room</h4>
                <p>View your creation displayed on a living room wall</p>
            </div>
            <div class="ar-option-card">
                <h4>Outdoor Display</h4>
                <p>Visualize your animation on an outdoor digital billboard</p>
            </div>
            <div class="ar-option-card">
                <h4>Mobile Device</h4>
                <p>See how your video appears on a smartphone or tablet</p>
            </div>
        </div>
        <div class="ar-note">
            <p><strong>Note:</strong> AR Preview is currently in beta. For best results, use videos with clear motion and high contrast.</p>
        </div>
    </div>
</div>
"""

# Custom CSS (from first code)
css = """
/* Base Styles */
:root {
    --primary-color: #4a6bff;
    --secondary-color: #5d3fd3;
    --accent-color: #ff6b6b;
    --accent-color-2: #38b2ac;
    --bg-color: #f8f9fa;
    --text-color: #333;
    --card-bg: white;
    --border-radius: 12px;
    --shadow: 0 8px 16px rgba(0, 0, 0, 0.1);
    --transition: all 0.3s ease;
    --gradient-1: linear-gradient(135deg, var(--primary-color), var(--secondary-color));
    --gradient-2: linear-gradient(135deg, #2d3748, #1a202c);
    --gradient-3: linear-gradient(135deg, #4299e1, #3182ce);
}

body {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, 'Open Sans', sans-serif;
    color: var(--text-color);
    background-color: var(--bg-color);
    line-height: 1.6;
}

.container {
    max-width: 1400px;
    margin: auto;
    padding: 0 20px;
}

/* Custom Header */
.custom-header {
    background: var(--gradient-1);
    color: white;
    padding: 40px 30px;
    border-radius: var(--border-radius);
    margin-bottom: 30px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    box-shadow: var(--shadow);
    position: relative;
    overflow: hidden;
}

.custom-header::before {
    content: '';
    position: absolute;
    top: 0;
    right: 0;
    bottom: 0;
    left: 0;
    background: url('data:image/svg+xml;utf8,<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100" viewBox="0 0 100 100"><rect fill="none" width="100" height="100"/><path d="M0,0 L100,100" stroke="rgba(255,255,255,0.05)" stroke-width="1"/></svg>');
    opacity: 0.3;
}

.header-content h1 {
    font-size: 3.2rem;
    margin: 0;
    font-weight: 800;
    text-shadow: 0 2px 10px rgba(0, 0, 0, 0.2);
}

.subtitle {
    font-size: 1.4rem;
    opacity: 0.9;
    margin-top: 10px;
    margin-bottom: 15px;
}

.header-badges {
    display: flex;
    gap: 10px;
    margin-top: 15px;
}

/* Animated Logo */
.animated-logo {
    position: relative;
    width: 100px;
    height: 100px;
}

.circle, .square, .triangle {
    position: absolute;
    animation-duration: 4s;
    animation-iteration-count: infinite;
    animation-timing-function: ease-in-out;
    filter: drop-shadow(0 2px 5px rgba(0, 0, 0, 0.2));
}

.circle {
    width: 40px;
    height: 40px;
    background-color: rgba(255, 255, 255, 0.8);
    border-radius: 50%;
    top: 10px;
    left: 30px;
    animation-name: float-circle;
}

.square {
    width: 35px;
    height: 35px;
    background-color: rgba(255, 255, 255, 0.8);
    bottom: 10px;
    left: 15px;
    animation-name: float-square;
}

.triangle {
    width: 0;
    height: 0;
    border-left: 20px solid transparent;
    border-right: 20px solid transparent;
    border-bottom: 40px solid rgba(255, 255, 255, 0.8);
    right: 5px;
    top: 30px;
    animation-name: float-triangle;
}

@keyframes float-circle {
    0%, 100% { transform: translateY(0) scale(1); }
    50% { transform: translateY(-15px) scale(1.1); }
}

@keyframes float-square {
    0%, 100% { transform: translateY(0) rotate(0deg); }
    50% { transform: translateY(15px) rotate(45deg); }
}

@keyframes float-triangle {
    0%, 100% { transform: translateX(0) rotate(0deg); }
    50% { transform: translateX(-15px) rotate(-10deg); }
}

/* Tabs Styling */
.tabs {
    margin-top: 20px;
}

.tab-nav {
    background-color: var(--card-bg);
    border-radius: var(--border-radius) var(--border-radius) 0 0;
    overflow: hidden;
    display: flex;
    box-shadow: 0 -2px 10px rgba(0, 0, 0, 0.05);
}

.tab-nav button {
    padding: 15px 25px;
    font-size: 16px;
    font-weight: 600;
    transition: var(--transition);
    border: none;
    background: transparent;
    cursor: pointer;
    flex: 1;
    text-align: center;
    position: relative;
    overflow: hidden;
}

.tab-nav button:hover {
    background-color: rgba(74, 107, 255, 0.1);
}

.tab-nav button.selected {
    background-color: var(--primary-color);
    color: white;
}

.tab-nav button.selected::after {
    content: '';
    position: absolute;
    bottom: 0;
    left: 0;
    right: 0;
    height: 3px;
    background-color: white;
}

/* Cards */
.card {
    background-color: var(--card-bg);
    border-radius: var(--border-radius);
    box-shadow: var(--shadow);
    transition: var(--transition);
    overflow: hidden;
    margin-bottom: 20px;
    border: 1px solid rgba(0, 0, 0, 0.05);
}

.card:hover {
    transform: translateY(-5px);
    box-shadow: 0 12px 20px rgba(0, 0, 0, 0.15);
}

.card-header {
    padding: 20px 25px;
    border-bottom: 1px solid rgba(0, 0, 0, 0.1);
    background-color: #fafafa;
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.card-title {
    font-size: 1.4rem;
    font-weight: 700;
    margin: 0;
    color: var(--primary-color);
    display: flex;
    align-items: center;
}

.card-title i {
    margin-right: 10px;
    font-size: 1.2rem;
}

.card-actions {
    display: flex;
    gap: 10px;
}

.card-content {
    padding: 25px;
}

/* Gradio Components */
.gradio-container {
    border-radius: var(--border-radius);
    box-shadow: none;
}

.gradio-slider {
    margin-top: 15px !important;
}

.gradio-checkbox {
    margin: 10px 0;
}

.gradio-dropdown {
    margin: 10px 0;
}

.gradio-slider input[type=range] {
    height: 6px;
    background: linear-gradient(to right, var(--primary-color), var(--secondary-color));
    border-radius: 3px;
}

.gradio-slider input[type=range]::-webkit-slider-thumb {
    background: var(--primary-color);
    box-shadow: 0 2px 5px rgba(0, 0, 0, 0.2);
}

/* Button Styles */
.primary-button {
    background: var(--gradient-1);
    color: white;
    border: none;
    padding: 12px 25px;
    border-radius: 30px;
    font-weight: 600;
    cursor: pointer;
    transition: var(--transition);
    box-shadow: 0 4px 10px rgba(74, 107, 255, 0.3);
    display: inline-flex;
    align-items: center;
    justify-content: center;
    gap: 8px;
}

.primary-button:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 15px rgba(74, 107, 255, 0.4);
}

.primary-button i {
    font-size: 0.9em;
}

.secondary-button {
    background-color: white;
    color: var(--primary-color);
    border: 2px solid var(--primary-color);
    padding: 10px 25px;
    border-radius: 30px;
    font-weight: 600;
    cursor: pointer;
    transition: var(--transition);
    display: inline-flex;
    align-items: center;
    justify-content: center;
    gap: 8px;
}

.secondary-button:hover {
    background-color: rgba(74, 107, 255, 0.1);
    transform: translateY(-2px);
}

.tertiary-button {
    background-color: transparent;
    color: var(--primary-color);
    border: none;
    padding: 10px 15px;
    border-radius: 8px;
    font-weight: 500;
    cursor: pointer;
    transition: var(--transition);
}

.tertiary-button:hover {
    background-color: rgba(74, 107, 255, 0.05);
}

/* Creative Toolkit */
.creative-toolkit {
    background-color: #fff;
    border-radius: var(--border-radius);
    box-shadow: var(--shadow);
    padding: 25px;
    margin: 20px 0;
    border-top: 4px solid var(--primary-color);
}

.creative-toolkit h3 {
    margin-top: 0;
    display: flex;
    align-items: center;
    gap: 10px;
    font-size: 1.5rem;
    color: var(--primary-color);
}

.toolkit-content {
    display: flex;
    flex-wrap: wrap;
    gap: 20px;
    margin-top: 15px;
}

.toolkit-card {
    flex: 1;
    min-width: 250px;
    background-color: #f8f9fc;
    border-radius: var(--border-radius);
    padding: 20px;
    border-left: 4px solid var(--primary-color);
    box-shadow: 0 3px 6px rgba(0, 0, 0, 0.05);
    transition: var(--transition);
}

.toolkit-card:hover {
    transform: translateY(-3px);
    box-shadow: 0 5px 10px rgba(0, 0, 0, 0.1);
}

.toolkit-card h4 {
    margin-top: 0;
    margin-bottom: 15px;
    color: var(--primary-color);
    font-size: 1.2rem;
    border-bottom: 1px solid rgba(0, 0, 0, 0.05);
    padding-bottom: 8px;
}

.toolkit-card ul {
    padding-left: 20px;
    margin-bottom: 0;
}

.toolkit-card li {
    margin-bottom: 10px;
}

.toolkit-examples {
    margin-top: 25px;
    padding-top: 20px;
    border-top: 1px solid rgba(0, 0, 0, 0.05);
}

.toolkit-examples h4 {
    margin-top: 0;
    margin-bottom: 15px;
    color: var(--primary-color);
}

.example-tags {
    display: flex;
    flex-wrap: wrap;
    gap: 12px;
}

.example-tag {
    background-color: rgba(74, 107, 255, 0.1);
    color: var(--primary-color);
    padding: 8px 15px;
    border-radius: 20px;
    font-size: 0.9rem;
    font-weight: 500;
    transition: var(--transition);
    cursor: pointer;
}

.example-tag:hover {
    background-color: rgba(74, 107, 255, 0.2);
    transform: translateY(-2px);
}

/* Tips & Insights */
.tips {
    background-color: #fff9f0;
    border-left: 5px solid #ffba44;
    padding: 20px;
    margin: 15px 0;
    border-radius: 0 var(--border-radius) var(--border-radius) 0;
    position: relative;
}

.tips::before {
    content: '💡';
    position: absolute;
    left: -15px;
    top: -10px;
    background-color: #ffba44;
    width: 30px;
    height: 30px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 16px;
}

.tips h4 {
    margin-top: 0;
    color: #b7791f;
}

/* Footer */
.custom-footer {
    background: var(--gradient-2);
    color: white;
    padding: 50px 30px 20px;
    border-radius: var(--border-radius);
    margin-top: 50px;
    position: relative;
    overflow: hidden;
}

.custom-footer::before {
    content: '';
    position: absolute;
    top: 0;
    right: 0;
    width: 300px;
    height: 300px;
    background: radial-gradient(circle, rgba(255,255,255,0.1) 0%, rgba(255,255,255,0) 70%);
    z-index: 1;
}

.footer-content {
    display: flex;
    flex-wrap: wrap;
    justify-content: space-between;
    max-width: 1200px;
    margin: 0 auto;
    position: relative;
    z-index: 2;
}

.footer-section {
    flex: 1;
    min-width: 250px;
    margin-bottom: 30px;
    padding: 0 15px;
}

.footer-section h3 {
    font-size: 1.2rem;
    margin-bottom: 20px;
    color: #63b3ed;
    position: relative;
    padding-bottom: 10px;
}

.footer-section h3::after {
    content: '';
    position: absolute;
    bottom: 0;
    left: 0;
    width: 40px;
    height: 3px;
    background-color: #63b3ed;
}

.footer-section ul {
    list-style: none;
    padding: 0;
    margin: 0;
}

.footer-section li {
    margin-bottom: 12px;
}

.footer-section a {
    color: #e2e8f0;
    text -webkit-link: none;
    text-decoration: none;
    transition: var(--transition);
    display: inline-flex;
    align-items: center;
    gap: 5px;
}

.footer-section a:hover {
    color: #63b3ed;
    transform: translateX(3px);
}

.footer-social {
    display: flex;
    gap: 15px;
    margin-top: 20px;
}

.footer-social a {
    background-color: rgba(255, 255, 255, 0.1);
    color: white;
    width: 36px;
    height: 36px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    transition: var(--transition);
}

.footer-social a:hover {
    background-color: #63b3ed;
    transform: translateY(-3px);
}

.newsletter-form {
    display: flex;
    margin-top: 15px;
}

.newsletter-input {
    flex: 1;
    padding: 10px 15px;
    border: none;
    border-radius: 5px 0 0 5px;
    background-color: rgba(255, 255, 255, 0.1);
    color: white;
    outline: none;
}

.newsletter-input::placeholder {
    color: rgba(255, 255, 255, 0.5);
}

.newsletter-button {
    background-color: #63b3ed;
    color: white;
    border: none;
    padding: 10px 15px;
    border-radius: 0 5px 5px 0;
    cursor: pointer;
    transition: var(--transition);
}

.newsletter-button:hover {
    background-color: #4299e1;
}

.footer-bar {
    text-align: center;
    border-top: 1px solid rgba(255, 255, 255, 0.1);
    padding-top: 20px;
    margin-top: 20px;
    font-size: 0.9rem;
    color: #a0aec0;
    display: flex;
    justify-content: space-between;
    align-items: center;
    flex-wrap: wrap;
    gap: 10px;
}

.footer-links {
    display: flex;
    gap: 20px;
    flex-wrap: wrap;
}

.footer-links a {
    color: #a0aec0;
    text-decoration: none;
    transition: var(--transition);
}

.footer-links a:hover {
    color: #63b3ed;
}

/* Feature Badge */
.feature-badge {
    display: inline-block;
    padding: 5px 12px;
    background-color: #ebf8ff;
    color: #3182ce;
    border-radius: 20px;
    font-size: 0.85rem;
    font-weight: 600;
    margin-left: 8px;
    box-shadow: 0 2px 5px rgba(0, 0, 0, 0.05);
    transition: var(--transition);
}

.feature-badge:hover {
    transform: translateY(-2px);
    box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
}

.new-badge {
    background-color: #feebef;
    color: #e53e3e;
}

/* User Gallery Section */
.user-gallery {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(220px, 1fr));
    gap: 25px;
    margin-top: 30px;
}

.gallery-item {
    background-color: white;
    border-radius: var(--border-radius);
    overflow: hidden;
    box-shadow: var(--shadow);
    transition: var(--transition);
    border: 1px solid rgba(0, 0, 0, 0.05);
}

.gallery-item:hover {
    transform: scale(1.03);
    box-shadow: 0 15px 25px rgba(0, 0, 0, 0.1);
}

.gallery-img {
    width: 100%;
    height: 180px;
    object-fit: cover;
    transition: var(--transition);
}

.gallery-item:hover .gallery-img {
    transform: scale(1.05);
}

.gallery-info {
    padding: 15px;
    border-top: 1px solid rgba(0, 0, 0, 0.05);
}

.gallery-title {
    font-weight: 600;
    margin: 0 0 5px 0;
    color: var(--text-color);
}

.gallery-author {
    font-size: 0.85rem;
    color: #718096;
    display: flex;
    align-items: center;
    gap: 5px;
}

.gallery-author img {
    width: 20px;
    height: 20px;
    border-radius: 50%;
}

/* Insights Section */
.insights-section {
    background-color: white;
    border-radius: var(--border-radius);
    padding: 30px;
    box-shadow: var(--shadow);
    margin-top: 40px;
    border-top: 4px solid var(--accent-color-2);
}

.insights-section h3 {
    margin-top: 0;
    color: var(--accent-color-2);
    font-size: 1.5rem;
    display: flex;
    align-items: center;
    gap: 10px;
}

.stat-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 25px;
    margin-top: 25px;
}

.stat-card {
    background-color: #f8f9fc;
    border-radius: var(--border-radius);
    padding: 25px;
    text-align: center;
    transition: var(--transition);
    border: 1px solid rgba(0, 0, 0, 0.05);
    position: relative;
    overflow: hidden;
}

.stat-card::before {
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 5px;
    background: var(--gradient-3);
}

.stat-card:hover {
    transform: translateY(-5px);
    box-shadow: 0 10px 20px rgba(0, 0, 0, 0.1);
}

.stat-value {
    font-size: 2.5rem;
    font-weight: 700;
    color: var(--primary-color);
    margin: 15px 0;
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 5px;
}

.stat-icon {
    font-size: 1.5rem;
    background-color: rgba(74, 107, 255, 0.1);
    width: 40px;
    height: 40px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
}

.stat-label {
    color: #718096;
    font-size: 1rem;
    font-weight: 500;
}

/* Custom Accordion */
.custom-accordion {
    background-color: white;
    border-radius: var(--border-radius);
    overflow: hidden;
    box-shadow: var(--shadow);
    margin-bottom: 25px;
    border: 1px solid rgba(0, 0, 0, 0.05);
}

.accordion-header {
    padding: 18px 25px;
    background-color: #f8f9fc;
    cursor: pointer;
    display: flex;
    justify-content: space-between;
    align-items: center;
    border-bottom: 1px solid #e2e8f0;
    transition: var(--transition);
}

.accordion-header:hover {
    background-color: #edf2f7;
}

.accordion-header h3 {
    margin: 0;
    font-size: 1.1rem;
    font-weight: 600;
    color: var(--primary-color);
    display: flex;
    align-items: center;
    gap: 10px;
}

.accordion-header h3 i {
    font-size: 1rem;
}

.accordion-indicator {
    transition: var(--transition);
}

.accordion-open .accordion-indicator {
    transform: rotate(180deg);
}

.accordion-body {
    padding: 25px;
    border-bottom: 1px solid #e2e8f0;
}

/* Image Comparison */
.comparison-container {
   Exercise display: flex;
    flex-direction: column;
    gap: 30px;
    margin-top: 30px;
}

.comparison-item {
    display: flex;
    flex-wrap: wrap;
    gap: 25px;
    background-color: white;
    border-radius: var(--border-radius);
    padding: 25px;
    box-shadow: var(--shadow);
    transition: var(--transition);
    border: 1px solid rgba(0, 0, 0, 0.05);
}

.comparison-image {
    flex: 1;
    min-width: 300px;
}

.comparison-image img {
    width: 100%;
    height: auto;
    border-radius: 8px;
}

.comparison-title {
    font-size: 1.2rem;
    font-weight: 600;
    margin-bottom: 10px;
    color: var(--primary-color);
}

@media (max-width: 768px) {
    .header-content h1 {
        font-size: 2.5rem;
    }

    .subtitle {
        font-size: 1.1rem;
    }

    .animated-logo {
        display: none;
    }

    .footer-section {
        flex: 100%;
    }
}
"""

# Create the Gradio interface (combined interface)
def create_interface():
    with gr.Blocks(css=css, title="MotionMaster Pro: AI Video Generator") as demo:
        gr.HTML(custom_header)
        with gr.Tabs() as tabs:
            # Single Image Tab (renamed from first code)
            with gr.TabItem("Image to Video (I2VGenXL)") as single_tab:
                with gr.Row():
                    with gr.Column():
                        with gr.Group():
                            gr.Markdown("## 📷 Input Image")
                            with gr.Row():
                                with gr.Column(scale=4):
                                    image_input = gr.Image(label="Upload your image", type="pil")
                                with gr.Column(scale=1):
                                    example_button = gr.Button("Load Example")
                            url_input = gr.Textbox(label="Or enter image URL")
                            suggested_prompts = gr.Textbox(label="Suggested Prompts (based on your image)", interactive=False)
                            suggest_button = gr.Button("👁️ Generate Prompt Suggestions")
                        with gr.Group():
                            gr.Markdown("## ✨ Motion Settings")
                            prompt = gr.Textbox(
                                label="Prompt (describe the motion you want)",
                                placeholder="Describe the motion you want to add to your image...",
                                lines=3
                            )
                            example_prompt = gr.Dropdown(
                                choices=example_prompts,
                                label="Need inspiration? Try one of these prompts",
                                interactive=True
                            )
                            negative_prompt = gr.Textbox(
                                label="Negative Prompt (what to avoid)",
                                value=default_negative_prompt,
                                lines=2
                            )
                            with gr.Accordion("Advanced Settings", open=False):
                                with gr.Row():
                                    with gr.Column():
                                        num_inference_steps = gr.Slider(
                                            minimum=10, maximum=50, value=25, step=1,
                                            label="Inference Steps"
                                        )
                                        guidance_scale = gr.Slider(
                                            minimum=1.0, maximum=15.0, value=9.0, step=0.1,
                                            label="Guidance Scale"
                                        )
                                    with gr.Column():
                                        seed = gr.Slider(
                                            minimum=-1, maximum=2147483647, step=1, value=-1,
                                            label="Seed (-1 for random)"
                                        )
                                        randomize_seed = gr.Button("🎲 Randomize Seed")
                        with gr.Group():
                            gr.Markdown("## 🎬 Effect Settings")
                            with gr.Row():
                                with gr.Column():
                                    effect_type = gr.Radio(
                                        ["none", "vintage", "dream", "cinematic"],
                                        label="Effect Type",
                                        value="none"
                                    )
                                with gr.Column():
                                    effect_intensity = gr.Slider(
                                        minimum=0.0, maximum=1.0, value=0.5, step=0.1,
                                        label="Effect Intensity"
                                    )
                            export_format = gr.Radio(
                                ["gif", "mp4"],
                                label="Export Format",
                                value="gif"
                            )
                            generate_button = gr.Button("🚀 Generate Video", variant="primary")
                    with gr.Column():
                        with gr.Group():
                            gr.Markdown("## 🎥 Output Video")
                            video_output = gr.Video(label="Generated Video")
                            image_output = gr.Image(label="Preview", visible=True)
                            output_message = gr.Textbox(label="Status")
                            with gr.Row():
                                download_button = gr.Button("💾 Download")
                                share_button = gr.Button("🌐 Share")
                        gr.HTML(ar_preview_html)
                        gr.HTML(creative_toolkit_html)

            # Batch Processing Tab (from first code)
            with gr.TabItem("Batch Processing") as batch_tab:
                with gr.Row():
                    with gr.Column():
                        batch_images = gr.Gallery(label="Upload Multiple Images")
                        batch_upload = gr.File(label="Upload Images", file_count="multiple")
                        batch_prompts = gr.Textbox(
                            label="Prompts (one per line, matching the number of images)",
                            placeholder="Enter one prompt per line, matching the number of uploaded images...",
                            lines=5
                        )
                        with gr.Accordion("Batch Settings", open=False):
                            batch_negative_prompt = gr.Textbox(
                                label="Negative Prompt (applies to all)",
                                value=default_negative_prompt,
                                lines=2
                            )
                            with gr.Row():
                                batch_steps = gr.Slider(
                                    minimum=10, maximum=50, value=25, step=1,
                                    label="Inference Steps"
                                )
                                batch_guidance = gr.Slider(
                                    minimum=1.0, maximum=15.0, value=9.0, step=0.1,
                                    label="Guidance Scale"
                                )
                            with gr.Row():
                                batch_effect = gr.Dropdown(
                                    ["none", "vintage", "dream", "cinematic"],
                                    label="Effect Type",
                                    value="none"
                                )
                                batch_effect_intensity = gr.Slider(
                                    minimum=0.0, maximum=1.0, value=0.5, step=0.1,
                                    label="Effect Intensity"
                                )
                            batch_format = gr.Radio(
                                ["gif", "mp4"],
                                label="Export Format",
                                value="gif"
                            )
                            batch_seed = gr.Slider(
                                minimum=-1, maximum=2147483647, step=1, value=-1,
                                label="Seed (-1 for random per image)"
                            )
                        batch_process_button = gr.Button("🔄 Process Batch", variant="primary")
                    with gr.Column():
                        batch_output = gr.Gallery(label="Generated Videos")
                        batch_message = gr.Textbox(label="Batch Status")
                        with gr.Row():
                            batch_download_all = gr.Button("💾 Download All")
                            batch_clear = gr.Button("🗑️ Clear Results")

            # Cinematic Video Generator Tab (from second code)
            with gr.TabItem("Cinematic Video Generator (SDXL)") as cinematic_tab:
                gr.Markdown("# Cinematic AI Video Generator with SDXL GIFs\nCreate a 1-minute cinematic video with animated GIF scenes and voiceover from image prompts in ~5-6 minutes using Colab L4 GPU (24 GB)")
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("## Input")
                        storyboard_file = gr.File(label="Upload Storyboard Text File", file_types=[".txt"])
                        gr.Markdown("**Format**: `Scene X: [desc]\nImage: [prompt]` (Voiceover uses Image prompt text)")
                        negative_prompt_cinematic = gr.Textbox(label="Negative Prompt", value="blurry, bad quality, deformed, ugly, low resolution, cartoonish, abstract", lines=2)
                        num_inference_steps_cinematic = gr.Slider(10, 50, 25, step=5, label="Inference Steps (Higher = More Detail)")
                        guidance_scale_cinematic = gr.Slider(1.0, 15.0, 7.5, step=0.5, label="Guidance Scale (Higher = More Prompt Adherence)")
                        seed_cinematic = gr.Number(label="Seed (-1 for random)", value=-1)
                        music_mood = gr.Radio(["cinematic", "dramatic", "upbeat", "suspense", "emotional"], label="Music Mood", value="cinematic")
                        create_btn = gr.Button("Generate 1-Minute Video", variant="primary")
                    with gr.Column():
                        gr.Markdown("## Output")
                        video_output_cinematic = gr.Video(label="Generated 1-Minute Cinematic Video")
                        storyboard_output = gr.Markdown(label="Parsed Storyboard")
                        seed_output = gr.Textbox(label="Used Seed")
                create_btn.click(
                    fn=generate_cinematic_video,
                    inputs=[storyboard_file, negative_prompt_cinematic, num_inference_steps_cinematic, guidance_scale_cinematic, seed_cinematic, music_mood],
                    outputs=[video_output_cinematic, storyboard_output, seed_output]
                )
                gr.Markdown("""
                ### Instructions:
                1. Upload a `.txt` file with your storyboard.
                2. The `Image:` text will be used for both animated GIF generation and voiceover.
                3. Click "Generate 1-Minute Video" (~5-6 minutes using preloaded SDXL).
                4. Download from `/content/outputs/`.
                - Each scene becomes a 12-frame GIF for a cinematic effect.
                - Models are preloaded on startup for efficiency.
                - Ensure GPU runtime is enabled (L4 recommended).
                """)

            # User Gallery & Analytics Tab (from first code)
            with gr.TabItem("User Gallery & Analytics") as analytics_tab:
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("## 📊 Usage Analytics")
                        with gr.Row():
                            with gr.Column():
                                total_gen_count = gr.Number(label="Total Generations")
                            with gr.Column():
                                unique_prompts = gr.Number(label="Unique Prompts")
                            with gr.Column():
                                success_rate = gr.Number(label="Success Rate (%)")
                        analytics_chart = gr.Image(label="Effects Usage")
                        refresh_analytics = gr.Button("🔄 Refresh Analytics")
                    with gr.Column():
                        gr.Markdown("## 🖼️ Your Recent Creations")
                        user_gallery = gr.Gallery(label="Recent Generations")
                        gallery_refresh = gr.Button("🔄 Refresh Gallery")

        gr.HTML(footer_html)

        # Component interactions for Single Image Tab
        example_button.click(
            lambda: random.choice(example_images),
            outputs=url_input
        )
        url_input.change(
            process_url_input,
            inputs=[url_input, prompt, negative_prompt, num_inference_steps, guidance_scale,
                    effect_type, effect_intensity, export_format, seed],
            outputs=[video_output, image_output, output_message]
        )
        suggest_button.click(
            suggest_prompts,
            inputs=[image_input],
            outputs=[suggested_prompts]
        )
        example_prompt.change(
            lambda x: x,
            inputs=[example_prompt],
            outputs=[prompt]
        )
        randomize_seed.click(
            lambda: random.randint(0, 2147483647),
            outputs=[seed]
        )
        generate_button.click(
            generate_video,
            inputs=[image_input, prompt, negative_prompt, num_inference_steps, guidance_scale,
                    effect_type, effect_intensity, export_format, seed],
            outputs=[video_output, image_output, output_message]
        )

        # Component interactions for Batch Processing Tab
        batch_upload.upload(
            lambda files: [f.name for f in files],
            inputs=[batch_upload],
            outputs=[batch_images]
        )
        batch_process_button.click(
            batch_process,
            inputs=[batch_images, batch_prompts, batch_negative_prompt, batch_steps, batch_guidance,
                    batch_effect, batch_effect_intensity, batch_format, batch_seed],
            outputs=[batch_output, batch_message]
        )

        # Component interactions for Analytics Tab
        refresh_analytics.click(
            lambda: (
                user_analytics["total_generations"],
                len(set(item["prompt"] for item in generation_history)),
                100 - (user_analytics["errors"] / max(1, user_analytics["total_generations"]) * 100)
            ),
            outputs=[total_gen_count, unique_prompts, success_rate]
        )
        refresh_analytics.click(
            generate_analytics_chart,
            outputs=[analytics_chart]
        )
        gallery_refresh.click(
            lambda: [item["path"] for item in sorted(generation_history, key=lambda x: x["timestamp"], reverse=True)[:12]],
            outputs=[user_gallery]
        )

    return demo

# Main execution
if __name__ == "__main__":
    # Preload SDXL model at startup
    preload_model()
    # Create and launch the interface
    app = create_interface()
    app.launch(share=True, debug=True)